# HTML Webpage Classification — Student, Faculty, Course, Project

This notebook trains a text classifier on `TRAIN.zip`, evaluates it using a validation split, predicts labels for every HTML file in `TEST.zip`, and writes the required `YOUR_ROLL_NO.txt` file.

**Model:** HTML text extraction + combined word/character TF-IDF + LinearSVC.

> Before running: place `TRAIN.zip` and `TEST.zip` in the same folder as this notebook, or change the paths in the next cell. If using Google Colab, upload both ZIP files when prompted.

## 1. Install and import libraries

In [ ]:
%pip install -q beautifulsoup4 scikit-learn pandas matplotlib

In [ ]:
import os
import re
import zipfile
import html
from pathlib import PurePosixPath
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
CLASSES = ["student", "faculty", "course", "project"]

# Edit this to your actual roll number before final submission.
ROLL_NO = "YOUR_ROLL_NO"

TRAIN_ZIP = "TRAIN.zip"
TEST_ZIP = "TEST.zip"

## 2. Load the ZIP files

The loader reads HTML files directly from the ZIP archives; manual extraction is not required.

If running in Google Colab and the ZIP files are not already present, uncomment and run the upload cell below.

In [ ]:
# OPTIONAL: Google Colab upload
# from google.colab import files
# uploaded = files.upload()  # Select TRAIN.zip and TEST.zip

In [ ]:
for path in [TRAIN_ZIP, TEST_ZIP]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Cannot find {path}. Put it beside this notebook "
            "or update TRAIN_ZIP / TEST_ZIP."
        )

print("TRAIN.zip size:", round(os.path.getsize(TRAIN_ZIP) / 1024, 1), "KB")
print("TEST.zip size :", round(os.path.getsize(TEST_ZIP) / 1024, 1), "KB")

## 3. Inspect the archive structure

Run this cell first. It prints sample paths so you can verify how the training labels are stored.

The main loader below expects each training HTML file to be inside a folder named after its class, for example `student/page.html`. It also supports paths such as `.../student/page.html`.

In [ ]:
def list_html_members(zip_path, limit=30):
    with zipfile.ZipFile(zip_path, "r") as z:
        members = [
            name for name in z.namelist()
            if name.lower().endswith((".html", ".htm"))
            and not name.endswith("/")
        ]
    print(f"{zip_path}: {len(members)} HTML files")
    for name in members[:limit]:
        print(" ", name)
    return members

train_members = list_html_members(TRAIN_ZIP)
test_members = list_html_members(TEST_ZIP)

## 4. Extract labels and preprocess HTML

**Label assumption:** the class name appears as one of the parent directory names in the training ZIP. The function checks every directory component, not just the immediate parent.

HTML preprocessing removes scripts, styles, navigation and other non-content elements, then extracts visible text. The page title is retained because it often contains useful class information.

In [ ]:
def label_from_archive_path(member_name):
    # ZIP paths conventionally use forward slashes.
    parts = [part.lower() for part in PurePosixPath(member_name).parts[:-1]]
    for part in reversed(parts):
        if part in CLASSES:
            return part
    return None


def decode_html(raw_bytes):
    # Try common encodings before falling back to replacement characters.
    for encoding in ("utf-8", "latin-1"):
        try:
            return raw_bytes.decode(encoding)
        except UnicodeDecodeError:
            pass
    return raw_bytes.decode("utf-8", errors="replace")


def clean_html(raw_html):
    soup = BeautifulSoup(raw_html, "html.parser")

    # Remove content that is usually not useful for page classification.
    for tag in soup([
        "script", "style", "noscript", "iframe",
        "svg", "nav", "footer"
    ]):
        tag.decompose()

    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    text = soup.get_text(separator=" ", strip=True)

    # Decode entities and normalize whitespace.
    text = html.unescape(title + " " + text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_training_data(zip_path):
    documents, labels, filenames = [], [], []
    skipped_no_label = []

    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.namelist():
            if not member.lower().endswith((".html", ".htm")):
                continue
            if member.endswith("/"):
                continue

            label = label_from_archive_path(member)
            if label is None:
                skipped_no_label.append(member)
                continue

            raw = decode_html(z.read(member))
            text = clean_html(raw)

            if text:
                documents.append(text)
                labels.append(label)
                filenames.append(PurePosixPath(member).name)

    if skipped_no_label:
        print(f"Warning: skipped {len(skipped_no_label)} HTML files "
              "because no class folder was found.")
        print("Examples:", skipped_no_label[:10])

    return documents, labels, filenames


X, y, train_filenames = load_training_data(TRAIN_ZIP)

print("Loaded training documents:", len(X))
print("Class distribution:")
display(pd.Series(y, name="class").value_counts().reindex(CLASSES, fill_value=0))

### If the loader reports that files have no labels

Inspect the paths printed in Section 3. If labels are stored in a separate file or encoded differently, update `label_from_archive_path()` to match that structure. Do not guess labels from the HTML content.

## 5. Check the data

In [ ]:
if len(X) == 0:
    raise ValueError(
        "No labeled training documents were loaded. "
        "Check the ZIP paths and label extraction rule."
    )

if len(set(y)) < 2:
    raise ValueError("At least two classes are needed to train a classifier.")

print("Example extracted text:")
print(X[0][:1000])

## 6. Split training data into training and validation sets

A stratified split keeps approximately the same class proportions in both sets. The validation set gives an estimate of performance on unseen documents.

In [ ]:
class_counts = Counter(y)
can_stratify = min(class_counts.values()) >= 2

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y if can_stratify else None
)

print("Training split:", len(X_train))
print("Validation split:", len(X_valid))

## 7. Build the TF-IDF + LinearSVC pipeline

- **Word TF-IDF (1–2 grams):** captures individual terms and short phrases.
- **Character TF-IDF (3–5 grams):** captures word fragments, names and spelling variations.
- **LinearSVC:** a strong and efficient classifier for sparse, high-dimensional text features.

The vectorizers are fitted only on the training split during validation, avoiding information leakage.

In [ ]:
word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=1,
    sublinear_tf=True,
    stop_words="english"
)

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=30000,
    min_df=1,
    sublinear_tf=True
)

features = FeatureUnion([
    ("word_tfidf", word_tfidf),
    ("char_tfidf", char_tfidf)
])

model = Pipeline([
    ("features", features),
    ("classifier", LinearSVC(C=1.0, class_weight=None))
])

model.fit(X_train, y_train)
print("Model training complete.")

## 8. Evaluate the validation predictions

In [ ]:
valid_predictions = model.predict(X_valid)

print("Validation accuracy:",
      round(accuracy_score(y_valid, valid_predictions), 4))

print("\nClassification report:")
print(classification_report(
    y_valid,
    valid_predictions,
    labels=CLASSES,
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_valid, valid_predictions, labels=CLASSES)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(cmap="Blues", values_format="d")
plt.title("Validation Confusion Matrix")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. Retrain on all labeled training documents

After validation, fit a fresh copy of the same pipeline using all available labeled training data. This uses the maximum training data for the final test predictions.

In [ ]:
final_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            max_features=50000,
            min_df=1,
            sublinear_tf=True,
            stop_words="english"
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            max_features=30000,
            min_df=1,
            sublinear_tf=True
        ))
    ])),
    ("classifier", LinearSVC(C=1.0, class_weight=None))
])

final_model.fit(X, y)
print("Final model trained on", len(X), "documents.")

## 10. Load TEST.zip HTML files and predict their classes

The test archive is unlabeled. We extract and preprocess every HTML document, then predict its class using the final model.

The output filename uses the basename (for example, `f1.html`) as in the exam's required format. If the ZIP contains duplicate basenames in different folders, check the warning and use relative paths if your evaluator expects them.

In [ ]:
def load_test_data(zip_path):
    documents, filenames, archive_paths = [], [], []

    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.namelist():
            if not member.lower().endswith((".html", ".htm")):
                continue
            if member.endswith("/"):
                continue

            raw = decode_html(z.read(member))
            text = clean_html(raw)

            documents.append(text)
            filenames.append(PurePosixPath(member).name)
            archive_paths.append(member)

    return documents, filenames, archive_paths


X_test, test_filenames, test_archive_paths = load_test_data(TEST_ZIP)

if not X_test:
    raise ValueError("No HTML documents found in TEST.zip.")

print("Loaded test documents:", len(X_test))

basename_counts = Counter(test_filenames)
duplicates = [name for name, count in basename_counts.items() if count > 1]
if duplicates:
    print("WARNING: duplicate basenames found:", duplicates[:20])
    print("If the evaluator distinguishes folders, write relative archive paths instead.")

In [ ]:
test_predictions = final_model.predict(X_test)

prediction_df = pd.DataFrame({
    "file_name": test_filenames,
    "predicted_class": test_predictions
})

display(prediction_df.head(10))
print("Predicted class distribution:")
display(prediction_df["predicted_class"].value_counts().reindex(CLASSES, fill_value=0))

## 11. Save the required text file

Each line is written as:

`file_name predicted_class_name`

There is no header row. Change `ROLL_NO` in Section 1 to your actual roll number before running this cell.

In [ ]:
output_filename = f"{ROLL_NO}.txt"

with open(output_filename, "w", encoding="utf-8", newline="\n") as output_file:
    for filename, prediction in zip(test_filenames, test_predictions):
        output_file.write(f"{filename} {prediction}\n")

print("Created:", output_filename)
print("Number of prediction lines:", len(test_predictions))

# Preview the first few lines
with open(output_filename, "r", encoding="utf-8") as f:
    for line in list(f)[:10]:
        print(line, end="")

## 12. Download the output file (Google Colab)

Run this optional cell in Google Colab to download the prediction text file.

In [ ]:
# OPTIONAL: Google Colab download
# from google.colab import files
# files.download(output_filename)

## 13. Explanation of the chosen approach

The task is a four-class text classification problem. The source documents are HTML pages, so the HTML markup is not directly suitable as input to a conventional text classifier. BeautifulSoup is used to remove scripts, styles and other non-content elements and extract the visible page text.

TF-IDF converts the text into numerical features. Word unigrams and bigrams represent important terms and short phrases, while character n-grams capture fragments of words, names and spelling variations. Combining these representations can help distinguish university pages whose categories are indicated by both terminology and page-writing patterns.

LinearSVC is selected because it performs well on sparse, high-dimensional text data and is computationally efficient. A stratified validation split is used to estimate generalization performance. Accuracy, precision, recall, F1-score and a confusion matrix are reported. Finally, the same pipeline is retrained on the complete labeled training dataset and used to classify the test HTML documents.

**Submission files:** submit this notebook (or the `.py` code) and the generated `<ROLL_NO>.txt` file. Verify that the output contains one line for every HTML file in `TEST.zip` and that the filename format matches the evaluator's expectations.